
# NFL 1H Total Model — V4 Master Workbook

This is the consolidated V4 notebook.

**Production benchmark:** V4.1  
**Challenger:** V4.2 stateful field-position model

The notebook is intentionally built around **NFL first-half totals only**. It contains:

- prior/current-season shrinkage
- first-half pace + points-per-drive baseline
- drive outcome probabilities: TD / FG / turnover / empty
- empirical first-half drive-count variance
- realistic 6/7/8-point TD scoring
- non-offensive / special-teams scoring residuals
- V4.1 Monte Carlo distribution
- V4.2 turnover → field-position state
- FanDuel odds / fair odds / EV helpers
- full-slate scanner
- one-game 1,000,000-simulation runner for a local PC

### Frozen benchmark from prior validation

V4.1 previously produced a **3-year ECE of 2.68%** across 2023–2025 line predictions. V4.2 is **not promoted yet**; it must beat or preserve V4.1 calibration in leakage-safe backtesting.

> Important: more simulations reduce Monte Carlo noise. They do **not** remove model error.


In [ ]:

# Run once in Colab or a fresh local environment.
# If packages are already installed, this is harmless.

%pip install -q nflreadpy polars pandas pyarrow numpy psutil


## 1. Imports and configuration

In [ ]:

import math
import os
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import nflreadpy as nfl
import psutil

# ----------------------------
# MODEL CONFIG
# ----------------------------
PRIOR_GAMES_EQUIVALENT = 6

FULL_SEASONS = list(range(2018, 2027))
CURRENT_SEASON = 2026
PRIOR_SEASON = 2025
CURRENT_WEEK = 2

TEAM_MAP = {
    "OAK": "LV",
    "STL": "LA",
    "SD": "LAC",
    "JAC": "JAX",
    "WSH": "WAS",
}

print("CPU:", platform.processor() or platform.machine())
print("Logical CPUs:", os.cpu_count())
print("RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 1))


## 2. Load and normalize NFL play-by-play

In [ ]:

pbp = nfl.load_pbp(FULL_SEASONS)

if isinstance(pbp, pl.LazyFrame):
    pbp = pbp.collect()

def normalize_team_codes(df):
    out = df
    for col in ["posteam", "defteam", "home_team", "away_team"]:
        if col in out.columns:
            out = out.with_columns(
                pl.col(col).replace(TEAM_MAP).alias(col)
            )
    return out

pbp = normalize_team_codes(pbp)

print("PBP shape:", pbp.shape)

display(
    pbp.group_by("season")
       .agg(pl.col("game_id").n_unique().alias("games"))
       .sort("season")
)


## 3. Build first-half drives, halftime targets, and field position

In [ ]:

# ---------------------------------------------------------
# FIRST-HALF DRIVE TABLE
# ---------------------------------------------------------

first_half_drives = (
    pbp
    .filter(
        (pl.col("qtr") <= 2) &
        pl.col("drive").is_not_null() &
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("half_seconds_remaining").is_not_null()
    )
    .with_columns([
        pl.col("play_type")
          .is_in(["run", "pass"])
          .cast(pl.Int8)
          .alias("_offensive_play"),

        (
            pl.col("play_type").is_in(["run", "pass"]) &
            (pl.col("epa").fill_null(0) > 0)
        ).cast(pl.Int8).alias("_successful_play"),

        pl.when(pl.col("play_type").is_in(["run", "pass"]))
          .then(pl.col("epa").fill_null(0))
          .otherwise(0)
          .alias("_offensive_epa"),

        (
            pl.col("play_type").is_in(["run", "pass"]) &
            (pl.col("yards_gained").fill_null(0) >= 20)
        ).cast(pl.Int8).alias("_explosive_play"),

        (
            (pl.col("interception").fill_null(0) == 1) |
            (pl.col("fumble_lost").fill_null(0) == 1)
        ).cast(pl.Int8).alias("_turnover_play"),

        pl.col("sack").fill_null(0).cast(pl.Int8).alias("_sack"),
        pl.col("pass_attempt").fill_null(0).cast(pl.Int8).alias("_dropback"),
        pl.col("first_down").fill_null(0).cast(pl.Int8).alias("_first_down"),

        (
            pl.col("play_type").is_in(["run", "pass"]) &
            (pl.col("yardline_100").fill_null(101) <= 20)
        ).cast(pl.Int8).alias("_red_zone_play"),
    ])
    .group_by(["game_id", "drive"])
    .agg([
        pl.col("season").first().alias("season"),
        pl.col("week").first().alias("week"),
        pl.col("season_type").first().alias("season_type"),
        pl.col("home_team").first().alias("home_team"),
        pl.col("away_team").first().alias("away_team"),
        pl.col("posteam").first().alias("posteam"),
        pl.col("defteam").first().alias("defteam"),

        pl.col("half_seconds_remaining").max().alias("drive_start_seconds"),
        pl.col("half_seconds_remaining").min().alias("drive_end_seconds"),

        pl.col("_offensive_play").sum().alias("offensive_plays"),
        pl.col("_successful_play").sum().alias("successful_plays"),
        pl.col("_offensive_epa").sum().alias("total_epa"),
        pl.col("_explosive_play").sum().alias("explosive_plays"),
        pl.col("_turnover_play").max().alias("turnover_drive"),
        pl.col("_sack").sum().alias("sacks"),
        pl.col("_dropback").sum().alias("dropbacks"),
        pl.col("_first_down").sum().alias("first_downs"),
        pl.col("_red_zone_play").max().alias("red_zone_trip"),

        pl.col("posteam_score").min().alias("start_score"),
        pl.col("posteam_score_post").max().alias("end_score"),
    ])
    .with_columns([
        (pl.col("drive_start_seconds") - pl.col("drive_end_seconds"))
            .alias("drive_seconds"),

        (pl.col("end_score") - pl.col("start_score"))
            .clip(lower_bound=0)
            .alias("drive_points"),
    ])
    .with_columns([
        (pl.col("drive_points") >= 6).cast(pl.Int8).alias("td_drive"),
        (pl.col("drive_points") == 3).cast(pl.Int8).alias("fg_drive"),

        pl.when(pl.col("drive_points") >= 6).then(pl.lit("TD"))
          .when(pl.col("drive_points") == 3).then(pl.lit("FG"))
          .when(pl.col("turnover_drive") == 1).then(pl.lit("TURNOVER"))
          .otherwise(pl.lit("EMPTY"))
          .alias("drive_outcome"),

        pl.when(pl.col("offensive_plays") > 0)
          .then(pl.col("total_epa") / pl.col("offensive_plays"))
          .otherwise(None)
          .alias("epa_per_play"),

        pl.when(pl.col("offensive_plays") > 0)
          .then(pl.col("successful_plays") / pl.col("offensive_plays"))
          .otherwise(None)
          .alias("success_rate"),
    ])
    .sort(["season", "week", "game_id", "drive"])
)

# ---------------------------------------------------------
# TRUE HALFTIME SCOREBOARD TARGET
# Includes defensive and special-teams scoring.
# ---------------------------------------------------------

halftime_totals = (
    pbp
    .filter(
        (pl.col("qtr") <= 2) &
        pl.col("total_home_score").is_not_null() &
        pl.col("total_away_score").is_not_null()
    )
    .group_by("game_id")
    .agg([
        pl.col("season").first().alias("season"),
        pl.col("week").first().alias("week"),
        pl.col("season_type").first().alias("season_type"),
        pl.col("home_team").first().alias("home_team"),
        pl.col("away_team").first().alias("away_team"),
        pl.col("total_home_score").max().alias("home_1h_points"),
        pl.col("total_away_score").max().alias("away_1h_points"),
    ])
    .with_columns(
        (pl.col("home_1h_points") + pl.col("away_1h_points"))
        .alias("actual_total")
    )
)

# ---------------------------------------------------------
# FIRST SCRIMMAGE SNAP = DRIVE START FIELD POSITION
# This intentionally removes kickoffs.
# ---------------------------------------------------------

drive_start_position = (
    pbp
    .filter(
        (pl.col("qtr") <= 2) &
        pl.col("drive").is_not_null() &
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("yardline_100").is_not_null() &
        pl.col("half_seconds_remaining").is_not_null() &
        pl.col("down").is_not_null()
    )
    .sort(
        ["game_id", "drive", "half_seconds_remaining"],
        descending=[False, False, True]
    )
    .group_by(["game_id", "drive"], maintain_order=True)
    .agg(
        pl.col("yardline_100").first().alias("start_yardline_100")
    )
)

drive_state = (
    first_half_drives
    .join(drive_start_position, on=["game_id", "drive"], how="left")
    .sort(["game_id", "drive"])
    .with_columns([
        pl.col("turnover_drive")
          .shift(1)
          .over("game_id")
          .fill_null(0)
          .alias("previous_drive_turnover"),

        pl.col("posteam")
          .shift(1)
          .over("game_id")
          .alias("previous_posteam"),
    ])
    .with_columns(
        (
            (pl.col("previous_drive_turnover") == 1) &
            (pl.col("previous_posteam") != pl.col("posteam"))
        ).cast(pl.Int8).alias("after_turnover")
    )
    .with_columns(
        pl.when(pl.col("start_yardline_100") <= 40)
          .then(pl.lit("SHORT"))
          .when(pl.col("start_yardline_100") <= 70)
          .then(pl.lit("NORMAL"))
          .when(pl.col("start_yardline_100") > 70)
          .then(pl.lit("BACKED_UP"))
          .otherwise(pl.lit("UNKNOWN"))
          .alias("field_position_bucket")
    )
)

print("Drive rows:", first_half_drives.height)
print("Halftime games:", halftime_totals.height)


## 4. Baseline pace + PPD model and shrinkage

In [ ]:

def build_team_ratings(drives_df):
    offense = (
        drives_df
        .group_by("posteam")
        .agg([
            pl.col("game_id").n_unique().alias("games"),
            pl.len().alias("drives"),
            pl.col("drive_points").sum().alias("points"),
            pl.col("offensive_plays").sum().alias("plays"),
            pl.col("drive_seconds").sum().alias("seconds"),
        ])
        .with_columns([
            (pl.col("drives") / pl.col("games")).alias("drives_per_1h"),
            (pl.col("points") / pl.col("drives")).alias("points_per_drive"),
            (pl.col("plays") / pl.col("drives")).alias("plays_per_drive"),
            (pl.col("seconds") / pl.col("drives")).alias("seconds_per_drive"),
        ])
        .rename({"posteam": "team"})
    )

    defense = (
        drives_df
        .group_by("defteam")
        .agg([
            pl.col("game_id").n_unique().alias("games"),
            pl.len().alias("drives_faced"),
            pl.col("drive_points").sum().alias("points_allowed"),
        ])
        .with_columns([
            (pl.col("drives_faced") / pl.col("games")).alias("opp_drives_per_1h"),
            (pl.col("points_allowed") / pl.col("drives_faced"))
                .alias("points_allowed_per_drive"),
        ])
        .rename({"defteam": "team"})
    )

    return offense, defense


def _blend_offense(prior_off, current_off, prior_strength):
    prior = prior_off.select([
        "team",
        pl.col("drives_per_1h").alias("prior_drives_per_1h"),
        pl.col("points_per_drive").alias("prior_points_per_drive"),
    ])

    if current_off is None or current_off.height == 0:
        return prior.with_columns([
            pl.lit(0).alias("current_games"),
            pl.col("prior_drives_per_1h").alias("drives_per_1h"),
            pl.col("prior_points_per_drive").alias("points_per_drive"),
        ])

    current = current_off.select([
        "team",
        pl.col("games").alias("current_games"),
        pl.col("drives_per_1h").alias("current_drives_per_1h"),
        pl.col("points_per_drive").alias("current_points_per_drive"),
    ])

    return (
        prior.join(current, on="team", how="left")
        .with_columns(pl.col("current_games").fill_null(0))
        .with_columns([
            pl.when(pl.col("current_games") > 0)
              .then(
                  (
                      pl.col("prior_drives_per_1h") * prior_strength +
                      pl.col("current_drives_per_1h") * pl.col("current_games")
                  ) / (prior_strength + pl.col("current_games"))
              )
              .otherwise(pl.col("prior_drives_per_1h"))
              .alias("drives_per_1h"),

            pl.when(pl.col("current_games") > 0)
              .then(
                  (
                      pl.col("prior_points_per_drive") * prior_strength +
                      pl.col("current_points_per_drive") * pl.col("current_games")
                  ) / (prior_strength + pl.col("current_games"))
              )
              .otherwise(pl.col("prior_points_per_drive"))
              .alias("points_per_drive"),
        ])
    )


def _blend_defense(prior_def, current_def, prior_strength):
    prior = prior_def.select([
        "team",
        pl.col("opp_drives_per_1h").alias("prior_opp_drives_per_1h"),
        pl.col("points_allowed_per_drive").alias("prior_points_allowed_per_drive"),
    ])

    if current_def is None or current_def.height == 0:
        return prior.with_columns([
            pl.lit(0).alias("current_games"),
            pl.col("prior_opp_drives_per_1h").alias("opp_drives_per_1h"),
            pl.col("prior_points_allowed_per_drive").alias("points_allowed_per_drive"),
        ])

    current = current_def.select([
        "team",
        pl.col("games").alias("current_games"),
        pl.col("opp_drives_per_1h").alias("current_opp_drives_per_1h"),
        pl.col("points_allowed_per_drive").alias("current_points_allowed_per_drive"),
    ])

    return (
        prior.join(current, on="team", how="left")
        .with_columns(pl.col("current_games").fill_null(0))
        .with_columns([
            pl.when(pl.col("current_games") > 0)
              .then(
                  (
                      pl.col("prior_opp_drives_per_1h") * prior_strength +
                      pl.col("current_opp_drives_per_1h") * pl.col("current_games")
                  ) / (prior_strength + pl.col("current_games"))
              )
              .otherwise(pl.col("prior_opp_drives_per_1h"))
              .alias("opp_drives_per_1h"),

            pl.when(pl.col("current_games") > 0)
              .then(
                  (
                      pl.col("prior_points_allowed_per_drive") * prior_strength +
                      pl.col("current_points_allowed_per_drive") * pl.col("current_games")
                  ) / (prior_strength + pl.col("current_games"))
              )
              .otherwise(pl.col("prior_points_allowed_per_drive"))
              .alias("points_allowed_per_drive"),
        ])
    )


def build_blended_ratings_for_week(
    drives_df,
    prior_season,
    current_season,
    week,
    prior_games_equivalent=PRIOR_GAMES_EQUIVALENT,
):
    prior_df = drives_df.filter(
        (pl.col("season") == prior_season) &
        (pl.col("season_type") == "REG")
    )

    current_df = drives_df.filter(
        (pl.col("season") == current_season) &
        (pl.col("season_type") == "REG") &
        (pl.col("week") < week)
    )

    prior_off, prior_def = build_team_ratings(prior_df)

    if current_df.height:
        current_off, current_def = build_team_ratings(current_df)
    else:
        current_off, current_def = None, None

    return (
        _blend_offense(prior_off, current_off, prior_games_equivalent),
        _blend_defense(prior_def, current_def, prior_games_equivalent),
    )


live_offense, live_defense = build_blended_ratings_for_week(
    first_half_drives,
    PRIOR_SEASON,
    CURRENT_SEASON,
    CURRENT_WEEK,
)


def project_matchup(
    home_team,
    away_team,
    offense_ratings=None,
    defense_ratings=None,
):
    if offense_ratings is None:
        offense_ratings = live_offense
    if defense_ratings is None:
        defense_ratings = live_defense

    home_off = offense_ratings.filter(pl.col("team") == home_team).row(0, named=True)
    away_off = offense_ratings.filter(pl.col("team") == away_team).row(0, named=True)
    home_def = defense_ratings.filter(pl.col("team") == home_team).row(0, named=True)
    away_def = defense_ratings.filter(pl.col("team") == away_team).row(0, named=True)

    home_drives = (home_off["drives_per_1h"] + away_def["opp_drives_per_1h"]) / 2
    away_drives = (away_off["drives_per_1h"] + home_def["opp_drives_per_1h"]) / 2

    home_ppd = (home_off["points_per_drive"] + away_def["points_allowed_per_drive"]) / 2
    away_ppd = (away_off["points_per_drive"] + home_def["points_allowed_per_drive"]) / 2

    home_points = home_drives * home_ppd
    away_points = away_drives * away_ppd

    return {
        "Home Team": home_team,
        "Away Team": away_team,
        "Home Expected Drives": float(home_drives),
        "Away Expected Drives": float(away_drives),
        "Home Expected PPD": float(home_ppd),
        "Away Expected PPD": float(away_ppd),
        "Home Projected 1H Points": float(home_points),
        "Away Projected 1H Points": float(away_points),
        "Projected 1H Total": float(home_points + away_points),
    }


## 5. Drive-outcome ratings for V4

In [ ]:

OUTCOME_ORDER = ["TD", "FG", "TURNOVER", "EMPTY"]


def build_drive_outcome_ratings(drives_df):
    d = drives_df.with_columns([
        (pl.col("drive_outcome") == "TD").cast(pl.Float64).alias("_td"),
        (pl.col("drive_outcome") == "FG").cast(pl.Float64).alias("_fg"),
        (pl.col("drive_outcome") == "TURNOVER").cast(pl.Float64).alias("_to"),
        (pl.col("drive_outcome") == "EMPTY").cast(pl.Float64).alias("_empty"),
    ])

    offense = (
        d.group_by("posteam")
         .agg([
             pl.col("game_id").n_unique().alias("games"),
             pl.len().alias("drives"),
             pl.col("_td").mean().alias("td_rate"),
             pl.col("_fg").mean().alias("fg_rate"),
             pl.col("_to").mean().alias("turnover_rate"),
             pl.col("_empty").mean().alias("empty_rate"),
         ])
         .rename({"posteam": "team"})
    )

    defense = (
        d.group_by("defteam")
         .agg([
             pl.col("game_id").n_unique().alias("games"),
             pl.len().alias("drives_faced"),
             pl.col("_td").mean().alias("td_rate_allowed"),
             pl.col("_fg").mean().alias("fg_rate_allowed"),
             pl.col("_to").mean().alias("turnover_rate_forced"),
             pl.col("_empty").mean().alias("empty_rate_allowed"),
         ])
         .rename({"defteam": "team"})
    )

    return offense, defense


def _blend_outcome_side(prior, current, metrics, prior_strength):
    prior_sel = prior.select(
        ["team"] + [pl.col(m).alias(f"prior_{m}") for m in metrics]
    )

    if current is None or current.height == 0:
        return prior_sel.with_columns(
            [pl.col(f"prior_{m}").alias(m) for m in metrics] +
            [pl.lit(0).alias("current_games")]
        )

    current_sel = current.select(
        ["team", pl.col("games").alias("current_games")] +
        [pl.col(m).alias(f"current_{m}") for m in metrics]
    )

    out = (
        prior_sel
        .join(current_sel, on="team", how="left")
        .with_columns(pl.col("current_games").fill_null(0))
    )

    exprs = []
    for m in metrics:
        exprs.append(
            pl.when(pl.col("current_games") > 0)
              .then(
                  (
                      pl.col(f"prior_{m}") * prior_strength +
                      pl.col(f"current_{m}") * pl.col("current_games")
                  ) / (prior_strength + pl.col("current_games"))
              )
              .otherwise(pl.col(f"prior_{m}"))
              .alias(m)
        )

    return out.with_columns(exprs)


def build_live_outcome_ratings(
    drives_df,
    prior_season=PRIOR_SEASON,
    current_season=CURRENT_SEASON,
    week=CURRENT_WEEK,
    prior_strength=PRIOR_GAMES_EQUIVALENT,
):
    prior_df = drives_df.filter(
        (pl.col("season") == prior_season) &
        (pl.col("season_type") == "REG")
    )

    current_df = drives_df.filter(
        (pl.col("season") == current_season) &
        (pl.col("season_type") == "REG") &
        (pl.col("week") < week)
    )

    prior_off, prior_def = build_drive_outcome_ratings(prior_df)

    if current_df.height:
        current_off, current_def = build_drive_outcome_ratings(current_df)
    else:
        current_off, current_def = None, None

    off_metrics = ["td_rate", "fg_rate", "turnover_rate", "empty_rate"]
    def_metrics = [
        "td_rate_allowed",
        "fg_rate_allowed",
        "turnover_rate_forced",
        "empty_rate_allowed",
    ]

    return (
        _blend_outcome_side(prior_off, current_off, off_metrics, prior_strength),
        _blend_outcome_side(prior_def, current_def, def_metrics, prior_strength),
    )


live_outcome_off, live_outcome_def = build_live_outcome_ratings(first_half_drives)


def get_matchup_drive_probs(offense_row, opponent_defense_row):
    probs = {
        "TD": (offense_row["td_rate"] + opponent_defense_row["td_rate_allowed"]) / 2,
        "FG": (offense_row["fg_rate"] + opponent_defense_row["fg_rate_allowed"]) / 2,
        "TURNOVER": (
            offense_row["turnover_rate"] +
            opponent_defense_row["turnover_rate_forced"]
        ) / 2,
        "EMPTY": (
            offense_row["empty_rate"] +
            opponent_defense_row["empty_rate_allowed"]
        ) / 2,
    }

    total = sum(probs.values())
    return {k: v / total for k, v in probs.items()}



## 6. Historical simulation distributions

V4 does **not** assume a Poisson half. We use historical first-half behavior:

- actual drive-count error relative to the baseline expectation
- 6 / 7 / 8-point TD scoring
- scoreboard points not captured by offensive drive scoring

The historical baseline feature builder is chronological so the drive-count residuals are not created with future information.


In [ ]:

def build_historical_baseline_features(
    drives_df,
    targets_df,
    start_season=2019,
    end_season=2025,
    prior_strength=PRIOR_GAMES_EQUIVALENT,
):
    rows = []

    target_lookup = {
        r["game_id"]: r["actual_total"]
        for r in targets_df
        .filter(
            (pl.col("season_type") == "REG") &
            (pl.col("season") >= start_season) &
            (pl.col("season") <= end_season)
        )
        .select(["game_id", "actual_total"])
        .iter_rows(named=True)
    }

    for season in range(start_season, end_season + 1):
        season_games = drives_df.filter(
            (pl.col("season") == season) &
            (pl.col("season_type") == "REG")
        )

        weeks = season_games.select("week").unique().sort("week")["week"].to_list()

        for week in weeks:
            off, defense = build_blended_ratings_for_week(
                drives_df,
                prior_season=season - 1,
                current_season=season,
                week=week,
                prior_games_equivalent=prior_strength,
            )

            games = (
                season_games
                .filter(pl.col("week") == week)
                .group_by(["game_id", "home_team", "away_team"])
                .agg(pl.len().alias("actual_total_drives"))
            )

            for g in games.iter_rows(named=True):
                home = g["home_team"]
                away = g["away_team"]

                ho = off.filter(pl.col("team") == home)
                ao = off.filter(pl.col("team") == away)
                hd = defense.filter(pl.col("team") == home)
                ad = defense.filter(pl.col("team") == away)

                if min(ho.height, ao.height, hd.height, ad.height) == 0:
                    continue

                ho = ho.row(0, named=True)
                ao = ao.row(0, named=True)
                hd = hd.row(0, named=True)
                ad = ad.row(0, named=True)

                home_drives = (ho["drives_per_1h"] + ad["opp_drives_per_1h"]) / 2
                away_drives = (ao["drives_per_1h"] + hd["opp_drives_per_1h"]) / 2

                home_ppd = (ho["points_per_drive"] + ad["points_allowed_per_drive"]) / 2
                away_ppd = (ao["points_per_drive"] + hd["points_allowed_per_drive"]) / 2

                expected_total_drives = home_drives + away_drives
                projected_total = home_drives * home_ppd + away_drives * away_ppd

                rows.append({
                    "season": season,
                    "week": week,
                    "game_id": g["game_id"],
                    "home_team": home,
                    "away_team": away,
                    "expected_total_drives": float(expected_total_drives),
                    "actual_total_drives": int(g["actual_total_drives"]),
                    "drive_error": float(g["actual_total_drives"] - expected_total_drives),
                    "projected_total": float(projected_total),
                    "actual_total": float(target_lookup.get(g["game_id"], np.nan)),
                })

    return pd.DataFrame(rows)


historical_baseline = build_historical_baseline_features(
    first_half_drives,
    halftime_totals,
    start_season=2019,
    end_season=CURRENT_SEASON - 1,
)

drive_errors = historical_baseline["drive_error"].dropna().to_numpy(dtype=float)

# ---------------------------------------------------------
# NON-OFFENSIVE / UNMODELED SCOREBOARD POINTS
# ---------------------------------------------------------

offensive_points_by_game = (
    first_half_drives
    .filter(
        (pl.col("season") < CURRENT_SEASON) &
        (pl.col("season_type") == "REG")
    )
    .group_by("game_id")
    .agg(pl.col("drive_points").sum().alias("offensive_drive_points"))
)

extra_scoring = (
    halftime_totals
    .filter(
        (pl.col("season") < CURRENT_SEASON) &
        (pl.col("season_type") == "REG")
    )
    .join(offensive_points_by_game, on="game_id", how="inner")
    .with_columns(
        (pl.col("actual_total") - pl.col("offensive_drive_points"))
        .clip(lower_bound=0)
        .alias("extra_points")
    )
)

extra_point_samples = extra_scoring["extra_points"].to_numpy()

# ---------------------------------------------------------
# REAL TD DRIVE SCORE DISTRIBUTION: 6 / 7 / 8
# ---------------------------------------------------------

td_history = (
    first_half_drives
    .filter(
        (pl.col("season") < CURRENT_SEASON) &
        (pl.col("season_type") == "REG") &
        pl.col("drive_points").is_in([6, 7, 8])
    )
)

td_counts = (
    td_history
    .group_by("drive_points")
    .agg(pl.len().alias("n"))
    .sort("drive_points")
)

td_values = td_counts["drive_points"].to_numpy()
td_probs = td_counts["n"].to_numpy().astype(float)
td_probs = td_probs / td_probs.sum()

print("Historical drive-error SD:", round(float(np.std(drive_errors)), 3))
print("TD values:", td_values)
print("TD probabilities:", np.round(td_probs, 5))
print("Extra-score sample size:", len(extra_point_samples))


## 7. V4.1 — frozen production benchmark

In [ ]:

def _live_matchup_outcome_probs(home_team, away_team):
    home_off = live_outcome_off.filter(pl.col("team") == home_team).row(0, named=True)
    away_off = live_outcome_off.filter(pl.col("team") == away_team).row(0, named=True)
    home_def = live_outcome_def.filter(pl.col("team") == home_team).row(0, named=True)
    away_def = live_outcome_def.filter(pl.col("team") == away_team).row(0, named=True)

    return (
        get_matchup_drive_probs(home_off, away_def),
        get_matchup_drive_probs(away_off, home_def),
    )


def _sample_outcomes_from_rows(rng, probs_2d):
    r = rng.random(probs_2d.shape[0])
    cdf = np.cumsum(probs_2d, axis=1)
    idx = (r[:, None] > cdf).sum(axis=1)
    return np.clip(idx, 0, probs_2d.shape[1] - 1)


def simulate_first_half_v41(
    home_team,
    away_team,
    n_simulations=50_000,
    seed=42,
):
    rng = np.random.default_rng(seed)

    baseline = project_matchup(home_team, away_team)
    expected_total_drives = (
        baseline["Home Expected Drives"] +
        baseline["Away Expected Drives"]
    )

    home_probs, away_probs = _live_matchup_outcome_probs(home_team, away_team)

    home_vec = np.array([home_probs[o] for o in OUTCOME_ORDER], dtype=float)
    away_vec = np.array([away_probs[o] for o in OUTCOME_ORDER], dtype=float)

    # Empirical possession-count error.
    total_drives = np.rint(
        expected_total_drives +
        rng.choice(drive_errors, size=n_simulations, replace=True)
    ).astype(np.int16)

    total_drives = np.clip(total_drives, 5, 20)

    # Randomize which team gets the first possession.
    home_first = rng.random(n_simulations) < 0.5

    scores = np.zeros(n_simulations, dtype=np.float32)

    # At most 20 vectorized drive steps.
    for drive_number in range(20):
        active = drive_number < total_drives
        if not active.any():
            break

        idx = np.flatnonzero(active)

        if drive_number % 2 == 0:
            is_home = home_first[idx]
        else:
            is_home = ~home_first[idx]

        probs = np.empty((len(idx), 4), dtype=np.float32)
        probs[is_home] = home_vec
        probs[~is_home] = away_vec

        outcomes = _sample_outcomes_from_rows(rng, probs)

        # 0 TD, 1 FG, 2 turnover, 3 empty
        td_mask = outcomes == 0
        if td_mask.any():
            scores[idx[td_mask]] += rng.choice(
                td_values,
                size=int(td_mask.sum()),
                p=td_probs,
            )

        fg_mask = outcomes == 1
        scores[idx[fg_mask]] += 3

    scores += rng.choice(
        extra_point_samples,
        size=n_simulations,
        replace=True,
    ).astype(np.float32)

    return {
        "home_team": home_team,
        "away_team": away_team,
        "baseline_projection": baseline["Projected 1H Total"],
        "expected_total_drives": expected_total_drives,
        "home_base_probs": home_probs,
        "away_base_probs": away_probs,
        "simulated_totals": scores,
    }


# Compatibility alias used by older cells.
simulate_first_half = simulate_first_half_v41


## 8. V4.2 — stateful field-position challenger

In [ ]:

FIELD_BUCKETS = ["SHORT", "NORMAL", "BACKED_UP"]

# V4.2 mechanism is estimated using completed seasons only.
field_history = (
    drive_state
    .filter(
        (pl.col("season") < CURRENT_SEASON) &
        (pl.col("season_type") == "REG") &
        (pl.col("field_position_bucket") != "UNKNOWN")
    )
)

# ---------------------------------------------------------
# FIELD-POSITION SANITY / RESEARCH TABLES
# ---------------------------------------------------------

field_position_results = (
    field_history
    .group_by("field_position_bucket")
    .agg([
        pl.len().alias("drives"),
        pl.col("start_yardline_100").mean().alias("avg_yards_to_goal"),
        pl.col("drive_points").mean().alias("points_per_drive"),
        (pl.col("drive_outcome") == "TD").mean().alias("td_rate"),
        (pl.col("drive_outcome") == "FG").mean().alias("fg_rate"),
        (pl.col("drive_outcome") == "TURNOVER").mean().alias("turnover_rate"),
    ])
    .sort("avg_yards_to_goal")
)

after_turnover_results = (
    field_history
    .group_by("after_turnover")
    .agg([
        pl.len().alias("drives"),
        pl.col("start_yardline_100").mean().alias("avg_yards_to_goal"),
        (pl.col("start_yardline_100") <= 40).mean().alias("short_field_rate"),
        pl.col("drive_points").mean().alias("points_per_drive"),
        (pl.col("drive_outcome") == "TD").mean().alias("td_rate"),
        (pl.col("drive_outcome") == "FG").mean().alias("fg_rate"),
    ])
    .sort("after_turnover")
)

print("SCORING BY STARTING FIELD POSITION")
display(field_position_results)

print("\nNEXT POSSESSION AFTER TURNOVER")
display(after_turnover_results)

# ---------------------------------------------------------
# LEAGUE OUTCOME RATES
# ---------------------------------------------------------

overall_outcome_rates = {}
for outcome in OUTCOME_ORDER:
    overall_outcome_rates[outcome] = (
        field_history.filter(pl.col("drive_outcome") == outcome).height
        / field_history.height
    )

field_outcome_rates = {}
for bucket in FIELD_BUCKETS:
    subset = field_history.filter(pl.col("field_position_bucket") == bucket)
    field_outcome_rates[bucket] = {
        outcome: (
            subset.filter(pl.col("drive_outcome") == outcome).height
            / subset.height
        )
        for outcome in OUTCOME_ORDER
    }

# ---------------------------------------------------------
# STARTING FIELD-POSITION DISTRIBUTION
# normal possession vs possession immediately after turnover
# ---------------------------------------------------------

field_bucket_probs = {}

for turnover_state in [0, 1]:
    subset = field_history.filter(pl.col("after_turnover") == turnover_state)

    counts = (
        subset.group_by("field_position_bucket")
              .agg(pl.len().alias("drives"))
    )

    total = counts["drives"].sum()
    prob_map = {bucket: 0.0 for bucket in FIELD_BUCKETS}

    for row in counts.iter_rows(named=True):
        bucket = row["field_position_bucket"]
        if bucket in prob_map:
            prob_map[bucket] = row["drives"] / total

    field_bucket_probs[turnover_state] = prob_map


def adjust_probs_for_field_position(base_probs, bucket):
    adjusted = {}

    for outcome in OUTCOME_ORDER:
        league_rate = overall_outcome_rates[outcome]
        bucket_rate = field_outcome_rates[bucket][outcome]

        multiplier = bucket_rate / league_rate if league_rate > 0 else 1.0
        adjusted[outcome] = base_probs[outcome] * multiplier

    total = sum(adjusted.values())
    return {o: adjusted[o] / total for o in OUTCOME_ORDER}


field_probs_normal = np.array(
    [field_bucket_probs[0][b] for b in FIELD_BUCKETS],
    dtype=np.float32,
)
field_probs_turnover = np.array(
    [field_bucket_probs[1][b] for b in FIELD_BUCKETS],
    dtype=np.float32,
)

field_probs_normal /= field_probs_normal.sum()
field_probs_turnover /= field_probs_turnover.sum()


def build_field_outcome_matrix(base_probs):
    rows = []
    for bucket in FIELD_BUCKETS:
        adjusted = adjust_probs_for_field_position(base_probs, bucket)
        row = np.array([adjusted[o] for o in OUTCOME_ORDER], dtype=np.float32)
        row /= row.sum()
        rows.append(row)
    return np.vstack(rows)


def simulate_first_half_v42_fast(
    home_team,
    away_team,
    n_simulations=50_000,
    seed=42,
):
    rng = np.random.default_rng(seed)

    baseline = project_matchup(home_team, away_team)
    expected_total_drives = (
        baseline["Home Expected Drives"] +
        baseline["Away Expected Drives"]
    )

    home_base_probs, away_base_probs = _live_matchup_outcome_probs(
        home_team, away_team
    )

    home_matrix = build_field_outcome_matrix(home_base_probs)
    away_matrix = build_field_outcome_matrix(away_base_probs)

    total_drives = np.rint(
        expected_total_drives +
        rng.choice(drive_errors, size=n_simulations, replace=True)
    ).astype(np.int16)

    total_drives = np.clip(total_drives, 5, 20)
    home_first = rng.random(n_simulations) < 0.5

    scores = np.zeros(n_simulations, dtype=np.float32)
    previous_turnover = np.zeros(n_simulations, dtype=bool)

    for drive_number in range(20):
        active = drive_number < total_drives
        if not active.any():
            break

        idx = np.flatnonzero(active)

        if drive_number % 2 == 0:
            is_home = home_first[idx]
        else:
            is_home = ~home_first[idx]

        turnover_state = previous_turnover[idx]

        selected_field_probs = np.where(
            turnover_state[:, None],
            field_probs_turnover[None, :],
            field_probs_normal[None, :],
        )

        field_idx = _sample_outcomes_from_rows(rng, selected_field_probs)

        outcome_probs = np.empty((len(idx), 4), dtype=np.float32)

        if is_home.any():
            outcome_probs[is_home] = home_matrix[field_idx[is_home]]

        if (~is_home).any():
            outcome_probs[~is_home] = away_matrix[field_idx[~is_home]]

        outcomes = _sample_outcomes_from_rows(rng, outcome_probs)

        td_mask = outcomes == 0
        if td_mask.any():
            scores[idx[td_mask]] += rng.choice(
                td_values,
                size=int(td_mask.sum()),
                p=td_probs,
            )

        fg_mask = outcomes == 1
        scores[idx[fg_mask]] += 3

        # A simulated turnover changes the field-position state
        # for the next possession.
        previous_turnover[idx] = outcomes == 2

    scores += rng.choice(
        extra_point_samples,
        size=n_simulations,
        replace=True,
    ).astype(np.float32)

    return {
        "home_team": home_team,
        "away_team": away_team,
        "baseline_projection": baseline["Projected 1H Total"],
        "expected_total_drives": expected_total_drives,
        "home_base_probs": home_base_probs,
        "away_base_probs": away_base_probs,
        "simulated_totals": scores,
    }


## 9. Odds, fair-price, and EV helpers

In [ ]:

def american_to_implied_prob(odds):
    if odds is None:
        return np.nan
    if odds < 0:
        return (-odds) / ((-odds) + 100)
    return 100 / (odds + 100)


def prob_to_american(prob):
    if prob <= 0 or prob >= 1:
        return np.nan
    if prob >= 0.5:
        return round(-100 * prob / (1 - prob))
    return round(100 * (1 - prob) / prob)


def profit_per_unit(odds):
    if odds < 0:
        return 100 / (-odds)
    return odds / 100


def ev_percent(model_prob, odds):
    return 100 * (
        model_prob * profit_per_unit(odds) -
        (1 - model_prob)
    )


def no_vig_two_way_prob(side_odds, other_side_odds):
    p1 = american_to_implied_prob(side_odds)
    p2 = american_to_implied_prob(other_side_odds)
    return p1 / (p1 + p2)


def evaluate_under_from_totals(totals, line, under_odds=None):
    p = float(np.mean(totals < line))

    result = {
        "line": line,
        "model_prob": p,
        "fair_odds": prob_to_american(p),
    }

    if under_odds is not None:
        implied = american_to_implied_prob(under_odds)
        result.update({
            "book_odds": under_odds,
            "book_break_even": implied,
            "probability_edge": p - implied,
            "EV_pct": ev_percent(p, under_odds),
        })

    return result


## 10. Analyze one matchup once, then query any alternate total

In [ ]:

def analyze_matchup_v4(
    home_team,
    away_team,
    lines=(22.5, 23.5, 24.5, 25.5),
    fanduel_under_odds=None,
    n_simulations=50_000,
    seed=42,
):
    # fanduel_under_odds example:
    # {22.5: -110, 23.5: -130, 24.5: -178, 25.5: -184}

    v41 = simulate_first_half_v41(
        home_team, away_team,
        n_simulations=n_simulations,
        seed=seed,
    )

    v42 = simulate_first_half_v42_fast(
        home_team, away_team,
        n_simulations=n_simulations,
        seed=seed + 1,
    )

    t41 = v41["simulated_totals"]
    t42 = v42["simulated_totals"]

    rows = []

    for line in lines:
        p41 = float(np.mean(t41 < line))
        p42 = float(np.mean(t42 < line))

        row = {
            "matchup": f"{away_team} @ {home_team}",
            "line": line,
            "baseline_projection": round(v41["baseline_projection"], 3),
            "v41_mean": round(float(t41.mean()), 3),
            "v42_mean": round(float(t42.mean()), 3),
            "v41_under_prob": p41,
            "v42_under_prob": p42,
            "v42_minus_v41": p42 - p41,
            "v41_fair_odds": prob_to_american(p41),
            "v42_fair_odds": prob_to_american(p42),
        }

        if fanduel_under_odds and line in fanduel_under_odds:
            odds = fanduel_under_odds[line]
            row.update({
                "fanduel_under_odds": odds,
                "break_even": american_to_implied_prob(odds),
                # V4.1 remains the official benchmark probability.
                "v41_EV_pct": ev_percent(p41, odds),
                "v42_EV_pct": ev_percent(p42, odds),
            })

        rows.append(row)

    return pd.DataFrame(rows), v41, v42


# EXAMPLE — GB @ MIN
GB_MIN_ODDS = {
    23.5: -130,
    24.5: -178,
    25.5: -184,
}

gb_min_table, gb_min_v41, gb_min_v42 = analyze_matchup_v4(
    home_team="MIN",
    away_team="GB",
    lines=[22.5, 23.5, 24.5, 25.5],
    fanduel_under_odds=GB_MIN_ODDS,
    n_simulations=50_000,
    seed=42,
)

display(gb_min_table)


## 11. Full-slate scanner — simulate each game once, evaluate many lines

In [ ]:

# Edit this slate as needed.
SUNDAY_GAMES = [
    ("CHI", "CAR"),
    ("BAL", "IND"),
    ("ATL", "PIT"),
    ("CLE", "JAX"),
    ("TB",  "CIN"),
    ("NYJ", "TEN"),
    ("NO",  "DET"),
    ("BUF", "HOU"),
    ("ARI", "LAC"),
    ("GB",  "MIN"),
    ("MIA", "LV"),
    ("WAS", "PHI"),
    ("DAL", "NYG"),
]

TEST_LINES = [23.5, 24.5, 25.5]


def run_v4_slate(
    games,
    lines=TEST_LINES,
    n_simulations=20_000,
    seed=42,
):
    rows = []

    for i, (away, home) in enumerate(games):
        print(f"Running {away} @ {home}...")

        v41 = simulate_first_half_v41(
            home, away,
            n_simulations=n_simulations,
            seed=seed + i,
        )

        v42 = simulate_first_half_v42_fast(
            home, away,
            n_simulations=n_simulations,
            seed=seed + 1000 + i,
        )

        t41 = v41["simulated_totals"]
        t42 = v42["simulated_totals"]

        for line in lines:
            p41 = float(np.mean(t41 < line))
            p42 = float(np.mean(t42 < line))

            rows.append({
                "matchup": f"{away} @ {home}",
                "line": line,
                "baseline_projection": v41["baseline_projection"],
                "v41_mean": float(t41.mean()),
                "v42_mean": float(t42.mean()),
                "v41_under_prob": p41,
                "v42_under_prob": p42,
                # Informational only until V4.2 passes backtesting.
                "consensus_avg": (p41 + p42) / 2,
                "model_difference": p42 - p41,
            })

    return pd.DataFrame(rows)


# Example:
# slate_results = run_v4_slate(
#     SUNDAY_GAMES,
#     lines=TEST_LINES,
#     n_simulations=20_000,
#     seed=42,
# )
#
# display(
#     slate_results.sort_values(
#         "v41_under_prob",
#         ascending=False
#     )
# )



## 12. Local-PC 1,000,000 simulation runner

Run games **one at a time**. A 32 GB machine is still plenty for this vectorized approach.

This saves the raw V4.1 and V4.2 distributions to an `.npz` file. Once saved, you can evaluate a new FanDuel alternate line instantly without rerunning the simulations.


In [ ]:

def run_million_and_save(
    home_team,
    away_team,
    lines=(20.5, 21.5, 22.5, 23.5, 24.5, 25.5, 26.5, 27.5),
    n_simulations=1_000_000,
    seed=4200,
    save_dir="v4_simulations",
):
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    start = time.time()

    print(f"Running V4.1: {away_team} @ {home_team}")
    v41 = simulate_first_half_v41(
        home_team,
        away_team,
        n_simulations=n_simulations,
        seed=seed,
    )

    print(f"Running V4.2: {away_team} @ {home_team}")
    v42 = simulate_first_half_v42_fast(
        home_team,
        away_team,
        n_simulations=n_simulations,
        seed=seed + 1,
    )

    t41 = v41["simulated_totals"]
    t42 = v42["simulated_totals"]

    rows = []
    for line in lines:
        rows.append({
            "line": line,
            "v41_under_prob": float(np.mean(t41 < line)),
            "v42_under_prob": float(np.mean(t42 < line)),
            "v41_fair_odds": prob_to_american(float(np.mean(t41 < line))),
            "v42_fair_odds": prob_to_american(float(np.mean(t42 < line))),
        })

    out = pd.DataFrame(rows)

    filename = f"{away_team}_{home_team}_{n_simulations:,}".replace(",", "") + ".npz"
    path = Path(save_dir) / filename

    np.savez_compressed(
        path,
        v41=t41,
        v42=t42,
        home_team=home_team,
        away_team=away_team,
        baseline_projection=v41["baseline_projection"],
    )

    elapsed = time.time() - start

    print("\nFinished.")
    print("Elapsed seconds:", round(elapsed, 2))
    print("V4.1 mean:", round(float(t41.mean()), 3))
    print("V4.2 mean:", round(float(t42.mean()), 3))
    print("Saved:", path)

    display(out)

    return out, path


# FIRST LOCAL MILLION-SIM TEST:
#
# million_table, million_file = run_million_and_save(
#     home_team="MIN",
#     away_team="GB",
#     n_simulations=1_000_000,
#     seed=4200,
# )


## 13. Load a saved million-sim game and price a new line instantly

In [ ]:

def load_saved_game(path):
    data = np.load(path, allow_pickle=True)
    return data["v41"], data["v42"]


def price_saved_under(path, line, under_odds=None):
    t41, t42 = load_saved_game(path)

    p41 = float(np.mean(t41 < line))
    p42 = float(np.mean(t42 < line))

    row = {
        "line": line,
        "v41_prob": p41,
        "v42_prob": p42,
        "v41_fair_odds": prob_to_american(p41),
        "v42_fair_odds": prob_to_american(p42),
    }

    if under_odds is not None:
        row.update({
            "book_odds": under_odds,
            "break_even": american_to_implied_prob(under_odds),
            "v41_EV_pct": ev_percent(p41, under_odds),
            "v42_EV_pct": ev_percent(p42, under_odds),
        })

    return row


# Example after running GB-MIN:
#
# print(
#     price_saved_under(
#         "v4_simulations/GB_MIN_1000000.npz",
#         line=23.5,
#         under_odds=-130,
#     )
# )



## 14. Model-version rules

### V4.1
Keep this as the **production benchmark** until something beats it out-of-sample.

### V4.2
Field position is a real football mechanism, but the model remains a challenger until it passes the same leakage-safe 2023–2025 calibration test.

### Live Test 1
For each game/line, record both probabilities **before kickoff**. Do not rewrite a prediction after seeing a result or a later market move.

### Simulation count
- 5k–20k: quick slate scan
- 50k: good live-analysis run
- 1M: remove almost all Monte Carlo noise from the final model estimate

One million simulations makes the simulator's estimate more precise. It does **not** make the underlying football assumptions automatically more accurate.
